In [3]:
"""
2026 Bamboo Summer AI Competition - improved RMSE pipeline
"""

import ast
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor, early_stopping, log_evaluation
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import MultiLabelBinarizer

DATA_DIR = Path(os.getenv("DATA_DIR", "/content"))
OUTPUT_PATH = Path(
    os.getenv("OUTPUT_PATH", str(DATA_DIR / "submission_improved.csv"))
)
N_SPLITS = int(os.getenv("N_SPLITS", "5"))
RANDOM_STATE = 42
REFERENCE_DATE = pd.Timestamp("2026-07-10")


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, np.clip(y_pred, 0, 1)) ** 0.5


def split_csv_items(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]


def split_languages(value):
    if pd.isna(value) or not str(value).strip():
        return []

    try:
        parsed = ast.literal_eval(str(value))
        values = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
    except (ValueError, SyntaxError):
        values = str(value).split(",")

    result = []
    for language in values:
        language = re.sub(r"<[^>]+>", "", str(language)).replace("*", "")
        language = re.sub(r"\s+", " ", language).strip()
        if language:
            result.append(language)
    return sorted(set(result))


def make_multihot(train_series, test_series, parser, label):
    encoder = MultiLabelBinarizer(sparse_output=True)
    train_items = train_series.apply(parser)
    test_items = test_series.apply(parser)

    train_matrix = encoder.fit_transform(train_items).astype(np.float32)
    known = set(encoder.classes_)
    test_items = test_items.apply(
        lambda items: [item for item in items if item in known]
    )
    test_matrix = encoder.transform(test_items).astype(np.float32)
    feature_names = [f"{label}_{x}" for x in encoder.classes_]

    print(f"{label} multi-hot features: {train_matrix.shape[1]}")
    return train_matrix, test_matrix, feature_names


def build_numeric_features(df):
    x = pd.DataFrame(index=df.index)

    price = pd.to_numeric(df["Price"], errors="coerce").fillna(0).clip(lower=0)
    discount = pd.to_numeric(df["Discount"], errors="coerce").fillna(0).clip(0, 100)
    required_age = (
        pd.to_numeric(df["Required age"], errors="coerce").fillna(0).clip(lower=0)
    )
    achievements = (
        pd.to_numeric(df["Achievements"], errors="coerce").fillna(0).clip(lower=0)
    )

    raw_numeric = {
        "price": price,
        "discount": discount,
        "required_age": required_age,
        "achievements": achievements,
    }
    for name, values in raw_numeric.items():
        x[name] = values
        x[f"log_{name}"] = np.log1p(values)
        x[f"has_{name}"] = (values > 0).astype("int8")

    release_date = pd.to_datetime(
        df["Release date"], errors="coerce", format="mixed"
    )
    days_since_release = (REFERENCE_DATE - release_date).dt.days

    x["release_year"] = release_date.dt.year.fillna(0)
    x["release_month"] = release_date.dt.month.fillna(0)
    x["release_day"] = release_date.dt.day.fillna(0)
    x["release_quarter"] = release_date.dt.quarter.fillna(0)
    x["release_dow"] = release_date.dt.dayofweek.fillna(-1)
    x["release_ordinal"] = release_date.map(
        lambda value: value.toordinal() if pd.notna(value) else 0
    )
    x["days_since_release"] = days_since_release.fillna(-1)
    x["log_days_since_release"] = np.log1p(
        days_since_release.fillna(0).clip(lower=0)
    )
    x["is_future_release"] = (
        (days_since_release < 0).fillna(False).astype("int8")
    )

    for platform in ["Windows", "Mac", "Linux"]:
        x[platform.lower()] = df[platform].astype("int8")
    x["n_platforms"] = x[["windows", "mac", "linux"]].sum(axis=1)

    languages = df["Supported languages"].apply(split_languages)
    categories = df["Categories"].apply(split_csv_items)
    genres = df["Genres"].apply(split_csv_items)
    x["n_languages"] = languages.str.len()
    x["n_categories"] = categories.str.len()
    x["n_genres"] = genres.str.len()

    about = df["About the game"].fillna("").astype(str)
    x["about_char_len"] = about.str.len()
    x["about_word_count"] = about.str.split().str.len()
    x["about_sentence_count"] = about.str.count(r"[.!?]")
    x["about_digit_count"] = about.str.count(r"\d")
    x["about_url_count"] = about.str.count(r"https?://|www\.")
    x["about_bracket_count"] = about.str.count(r"\[|\]")
    x["about_non_ascii"] = about.apply(
        lambda text: sum(ord(char) > 127 for char in text)
    )

    log_columns = [
        "about_char_len",
        "about_word_count",
        "about_sentence_count",
        "about_digit_count",
        "about_non_ascii",
    ]
    for column in log_columns:
        x[f"log_{column}"] = np.log1p(x[column])

    x["missing_about"] = about.str.strip().eq("").astype("int8")
    x["missing_categories"] = df["Categories"].isna().astype("int8")
    x["missing_genres"] = df["Genres"].isna().astype("int8")

    # Restore actual Steam App ID and image update timestamp from Header image URL
    header = df["Header image"].fillna("").astype(str)
    steam_appid = pd.to_numeric(
        header.str.extract(r"/apps/(\d+)/", expand=False), errors="coerce"
    )
    image_timestamp = pd.to_numeric(
        header.str.extract(r"[?&]t=(\d+)", expand=False), errors="coerce"
    )
    image_date = pd.to_datetime(image_timestamp, unit="s", errors="coerce")

    x["steam_appid"] = steam_appid.fillna(-1)
    x["log_steam_appid"] = np.log1p(steam_appid.fillna(0))
    x["image_update_year"] = image_date.dt.year.fillna(0)
    x["image_update_month"] = image_date.dt.month.fillna(0)
    x["image_update_ordinal"] = image_date.map(
        lambda value: value.toordinal() if pd.notna(value) else 0
    )
    x["days_release_to_image_update"] = (
        (image_date - release_date).dt.days.fillna(-1).clip(-1, 10000)
    )
    x["missing_header"] = header.eq("").astype("int8")

    x["discounted_price_proxy"] = price * (1 - discount / 100)
    x["price_per_year"] = price / (
        1 + days_since_release.fillna(0).clip(lower=0) / 365.25
    )

    return (
        x.replace([np.inf, -np.inf], 0)
        .fillna(0)
        .astype(np.float32)
    )


def make_regressor(seed):
    return LGBMRegressor(
        objective="regression_l2",
        n_estimators=2400,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
        force_row_wise=True,
    )


def make_classifier(seed):
    return LGBMClassifier(
        objective="multiclass",
        n_estimators=1600,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


def main():
    train = pd.read_csv(DATA_DIR / "/content/drive/MyDrive/kaggle/SteamBamboo/train.csv")
    test = pd.read_csv(DATA_DIR / "/content/drive/MyDrive/kaggle/SteamBamboo/test.csv")
    sample = pd.read_csv(DATA_DIR / "/content/drive/MyDrive/kaggle/SteamBamboo/sample_submission.csv")
    y = train["popularity_score"].to_numpy(np.float32)
    n_train = len(train)

    # Concatenate and split numeric features to ensure identical transformations
    combined_numeric = build_numeric_features(
        pd.concat([train.drop(columns=["popularity_score"]), test], ignore_index=True)
    )
    numeric_train = combined_numeric.iloc[:n_train]
    numeric_test = combined_numeric.iloc[n_train:]

    genre_train, genre_test, _ = make_multihot(
        train["Genres"], test["Genres"], split_csv_items, "genre"
    )
    category_train, category_test, _ = make_multihot(
        train["Categories"], test["Categories"], split_csv_items, "category"
    )
    language_train, language_test, _ = make_multihot(
        train["Supported languages"],
        test["Supported languages"],
        split_languages,
        "language",
    )

    x_train = hstack(
        [
            csr_matrix(numeric_train.to_numpy()),
            genre_train,
            category_train,
            language_train,
        ],
        format="csr",
    )
    x_test = hstack(
        [
            csr_matrix(numeric_test.to_numpy()),
            genre_test,
            category_test,
            language_test,
        ],
        format="csr",
    )
    print("Final feature shapes:", x_train.shape, x_test.shape)

    # Most frequent two point masses: 0 and 0.195695
    nonzero_counts = train.loc[
        train["popularity_score"] != 0, "popularity_score"
    ].value_counts()
    special_value = float(nonzero_counts.index[0])
    target_class = np.full(n_train, 2, dtype=np.int8)
    target_class[np.isclose(y, 0.0, atol=1e-8)] = 0
    target_class[np.isclose(y, special_value, atol=5e-7)] = 1
    print(
        "Target class counts:",
        dict(zip(*np.unique(target_class, return_counts=True))),
        "special value:",
        special_value,
    )

    oof_base = np.zeros(n_train, dtype=np.float32)
    oof_three_part = np.zeros(n_train, dtype=np.float32)
    test_base = np.zeros(len(test), dtype=np.float32)
    test_three_part = np.zeros(len(test), dtype=np.float32)

    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (train_idx, valid_idx) in enumerate(kfold.split(x_train), start=1):
        print(f"\n===== Fold {fold}/{N_SPLITS} =====")

        base_model = make_regressor(RANDOM_STATE + fold)
        base_model.fit(
            x_train[train_idx],
            y[train_idx],
            eval_set=[(x_train[valid_idx], y[valid_idx])],
            eval_metric="rmse",
            callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
        )
        base_valid = np.clip(base_model.predict(x_train[valid_idx]), 0, 1)
        oof_base[valid_idx] = base_valid
        test_base += (
            np.clip(base_model.predict(x_test), 0, 1).astype(np.float32)
            / N_SPLITS
        )

        class_model = make_classifier(RANDOM_STATE + 100 + fold)
        class_model.fit(
            x_train[train_idx],
            target_class[train_idx],
            eval_set=[(x_train[valid_idx], target_class[valid_idx])],
            eval_metric="multi_logloss",
            callbacks=[early_stopping(120, verbose=False), log_evaluation(0)],
        )

        other_train_idx = train_idx[target_class[train_idx] == 2]
        other_valid_mask = target_class[valid_idx] == 2
        other_model = make_regressor(RANDOM_STATE + 200 + fold)
        other_model.fit(
            x_train[other_train_idx],
            y[other_train_idx],
            eval_set=[
                (
                    x_train[valid_idx][other_valid_mask],
                    y[valid_idx][other_valid_mask],
                )
            ],
            eval_metric="rmse",
            callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
        )

        valid_probability = class_model.predict_proba(x_train[valid_idx])
        valid_other = np.clip(other_model.predict(x_train[valid_idx]), 0, 1)
        three_part_valid = (
            valid_probability[:, 1] * special_value
            + valid_probability[:, 2] * valid_other
        )
        oof_three_part[valid_idx] = three_part_valid.astype(np.float32)

        test_probability = class_model.predict_proba(x_test)
        test_other = np.clip(other_model.predict(x_test), 0, 1)
        test_three_part += (
            test_probability[:, 1] * special_value
            + test_probability[:, 2] * test_other
        ).astype(np.float32) / N_SPLITS

        print(f"base fold RMSE       : {rmse(y[valid_idx], base_valid):.8f}")
        print(
            f"three-part fold RMSE : "
            f"{rmse(y[valid_idx], three_part_valid):.8f}"
        )

    # Calculate optimal least squares blending weights from OOF predictions
    direction = oof_three_part - oof_base
    denominator = float(np.dot(direction, direction))
    if denominator > 0:
        three_part_weight = float(
            np.clip(np.dot(y - oof_base, direction) / denominator, 0, 1)
        )
    else:
        three_part_weight = 1.0
    base_weight = 1.0 - three_part_weight

    oof_final = np.clip(
        base_weight * oof_base + three_part_weight * oof_three_part, 0, 1
    )
    test_final = np.clip(
        base_weight * test_base + three_part_weight * test_three_part, 0, 1
    )

    print("\n===== OOF Results =====")
    print(f"base OOF RMSE       : {rmse(y, oof_base):.8f}")
    print(f"three-part OOF RMSE : {rmse(y, oof_three_part):.8f}")
    print(
        f"Optimal weights     : base={base_weight:.4f}, "
        f"three-part={three_part_weight:.4f}"
    )
    print(f"final OOF RMSE      : {rmse(y, oof_final):.8f}")

    submission = sample.copy()
    assert len(submission) == len(test)
    assert np.array_equal(submission["ID"].to_numpy(), test["ID"].to_numpy())
    submission["popularity_score"] = test_final
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    print("Saved submission file:", OUTPUT_PATH)


if __name__ == "__main__":
    main()

genre multi-hot features: 33
category multi-hot features: 58
language multi-hot features: 251
Final feature shapes: (100684, 394) (25171, 394)
Target class counts: {np.int8(0): np.int64(19630), np.int8(1): np.int64(12571), np.int8(2): np.int64(68483)} special value: 0.195695

===== Fold 1/5 =====


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


base fold RMSE       : 0.07125487
three-part fold RMSE : 0.07058888

===== Fold 2/5 =====


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


base fold RMSE       : 0.07004588
three-part fold RMSE : 0.06936698

===== Fold 3/5 =====


...

base fold RMSE       : 0.07141524
three-part fold RMSE : 0.07074454

===== Fold 4/5 =====


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


base fold RMSE       : 0.07102971
three-part fold RMSE : 0.07027402

===== Fold 5/5 =====


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


base fold RMSE       : 0.07158499
three-part fold RMSE : 0.07077988

===== OOF Results =====
base OOF RMSE       : 0.07106820
three-part OOF RMSE : 0.07035280
Optimal weights     : base=0.2432, three-part=0.7568
final OOF RMSE      : 0.07026996
Saved submission file: /content/submission_improved.csv


In [4]:
re = pd.read_csv('/content/submission_improved.csv')
re.head()

,ID,popularity_score
0,1087706,0.000070
1,1463177,0.349333
2,6716809,0.323102
3,1506824,0.000028
4,9902666,0.290395
